# 03 · Policy node — which clauses apply, and what they decide

**What this notebook does:** adds the third node of RM Copilot. It reads `signals` (and the charge
records behind them), checks **every clause in the policy corpus**, and writes the result: which clauses
apply and on what evidence, the overall outcome under EVD-07 (DECLINE > INSUFFICIENT EVIDENCE > REFER >
PROCEED), whether the company qualifies as a lead, and what evidence is still missing.

**What it deliberately does not do:** write prose, or decide whether to go back for more research. It
lists the gaps; the supervisor (notebook 04) decides what to do about them, and the brief (04) turns the
clauses into sentences.

**The plan said "rules selected in code, applicability judged by the model".** After notebook 02, most of
that judgement has nothing left to judge: every *Applies when* line is now a lookup on a signal, and a model
re-deriving `264 > 180` can only add error. So this node splits the work three ways:

| the record… | who answers | example |
|---|---|---|
| settles the clause | **code** — a rule per clause | CON-02: latest accounts 264 days late → applies |
| holds text that needs reading | **the model**, with a checked quote | SEC-07 on a pre-2013 charge whose pledge flag was never recorded |
| doesn't contain the answer | **nobody** — it becomes an evidence gap | CON-03 when the filing window was cut short |

A clause with no rule written for it at all goes to the model too, so a new clause added to the policy files
is never silently ignored (section 12). For 36EL the record settles everything, so the node makes zero model calls.

## 1 · Setup

`policy_store.py` (extracted from `06_rag.ipynb`) is imported for its clause parser, `load_clauses`, so the clause
text and each clause's outcome come from the policy files themselves, never from this notebook.

In [8]:
import json, sys
from pathlib import Path
from typing import Literal, TypedDict

import anthropic
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

ROOT  = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists())
GENAI = ROOT / "genai"
load_dotenv(ROOT / ".env")
sys.path.insert(0, str(GENAI))                  # so `import policy_store` resolves
from policy_store import load_clauses

print("repo root:", ROOT)

repo root: /Users/natchalin_/Projects/final_project/Lloyds


## 2 · The state — five new slots

| slot | type | read by |
|---|---|---|
| `applicable` | list of `{clause_id, title, outcome, evidence, decided_by}` | brief — every applying clause must be cited (EVD-06, EVD-07) |
| `outcome` | `{"decision": "DECLINE", "clauses": ["CON-02"]}` | brief — names the clause that set the outcome (EVD-06) |
| `qualifies` | `True` / `False` / `None` | the lead list, and the supervisor's routing |
| `evidence_gap` | list of strings; empty = no gap | supervisor — decides whether to go back to research (EVD-05) |
| `policy_trace` | `{clause_id: verdict}` for *every* clause | audit — shows each clause was checked, not just the ones that fired |

**`qualifies` has three values on purpose.** `False` = DECLINE (a gap can't rescue it, DECLINE is already
the most restrictive). `None` = can't say yet, because there's an evidence gap. `True` = PROCEED or REFER
with no gap; a REFER is still a company the RM can approach, with conditions.

The node reads `signals` plus `charges`. It needs `charges` for one thing only: the raw particulars text of a
charge, when the model has to read it.

In [9]:
class CopilotState(TypedDict, total=False):
    company_number: str
    as_of:    str
    profile:  dict              # ┐
    filings:  dict              # │ research
    charges:  dict              # │
    officers: dict              # ┘
    signals:  dict              # signal
    applicable:   list[dict]    # ┐
    outcome:      dict          # │
    qualifies:    bool | None   # │ policy fills these five
    evidence_gap: list[str]     # │
    policy_trace: dict          # ┘

## 3 · Start from the state notebook 02 saved

Notebook 02 ends by writing its final state to `production/state/`. Loading that here gives the policy node
exactly the input the graph would give it: the same records, the same signals, the same `as_of`. It also
means this notebook needs no MCP server and doesn't repeat 02's model call. Copying research and signal in here,
as 02 copied research from 01, would be about 250 lines to keep in sync.

In [10]:
STATE_DIR = GENAI / "production" / "state"
FIXTURES = {n: json.loads((STATE_DIR / f"after_signal_{n}.json").read_text())
            for n in ("10812571", "00445790")}

for n, st in FIXTURES.items():
    print(f"{n}  {st['profile']['company_name']:10s}  as_of={st['as_of']}  slots: {list(st)}")

10812571  36EL LTD    as_of=2026-09-23  slots: ['company_number', 'as_of', 'profile', 'filings', 'charges', 'officers', 'signals']
00445790  TESCO PLC   as_of=2026-09-23  slots: ['company_number', 'as_of', 'profile', 'filings', 'charges', 'officers', 'signals']


## 4 · The corpus — and which clauses this node owns

`load_clauses()` splits the three policy files into 23 clauses, one per `### ` heading, each with its
`Outcome:` line parsed out. Not all 23 are decided here. Some are rules about *writing* the brief, one is about the
retry loop, and one is the precedence rule this node implements. Those are **routed**: recorded in the trace
with where they're enforced, so it's clear they weren't skipped.

| routed clause | why not here | enforced in |
|---|---|---|
| CON-08 paper filing | a presentation rule ("must not be presented as a concern"); PROCEED can never change the outcome. `paper_filed` isn't retrieved either (notebook 02). | brief |
| EVD-02, EVD-03, EVD-06 | rules on what a brief may assert and cite | brief |
| EVD-04 | applies when the brief *proposes* a collateral statement; `signals.collateral[*].verified` says which it may make | brief |
| EVD-05 | a limit on research cycles, the supervisor's counter | supervisor |
| EVD-07 | the precedence rule itself | this node, section 7 |

In [11]:
CLAUSES = load_clauses()

ROUTED = {
    "CON-08": "brief",
    "EVD-02": "brief", "EVD-03": "brief", "EVD-04": "brief", "EVD-06": "brief",
    "EVD-05": "supervisor",
    "EVD-07": "this node (precedence)",
}

for c in CLAUSES:
    print(f"{c['clause_id']:7s} {c['outcome'][:26]:28s} {c['title'].split('—', 1)[1].strip()[:45]:47s}"
          f"{'→ ' + ROUTED[c['clause_id']] if c['clause_id'] in ROUTED else ''}")

CON-01  REFER                        Late filing of annual accounts                 
CON-02  DECLINE                      Materially late filing                         
CON-03  REFER                        Pattern of late filing                         
CON-04  REFER                        Micro-entity accounts                          
CON-05  DECLINE                      No accounts on record                          
CON-06  REFER                        Stale financial information                    
CON-07  DECLINE                      Insolvency proceedings                         
CON-08  PROCEED                      Paper filing                                   → brief
EVD-01  INSUFFICIENT EVIDENCE — re   Minimum evidence set                           
EVD-02  mandatory                    Every assertion must be sourced                → brief
EVD-03  mandatory                    No external knowledge                          → brief
EVD-04  INSUFFICIENT EVIDENCE — re   Charge 

## 5 · Rules in code — one per clause

Each rule reads the signals and gives one of four answers:

- `decided(True, evidence)` / `decided(False)`: the record settles it.
- `unknown(why)`: the record doesn't contain the answer, e.g. CON-03 when the filing window was truncated. This
  becomes an evidence gap. No model call, since a model can't read what isn't there.
- `read(targets)`: the record holds text that needs reading, e.g. a live pre-2013 charge whose pledge and floating
  flags were never recorded, but whose particulars might say. Those go to the model (section 6).

Most rules are the signal list itself (`_listed`). Notebook 02 already shaped each signal as "empty list = does not
apply; non-empty = applies, and here is the evidence".

`charge_refs` is **copied from notebook 02** because the model needs to find a charge's record from its reference.
If the two copies ever drift apart, the check at the top of `evaluate` (section 8) fails loudly.

In [12]:
def charge_refs(items: list[dict]) -> list[str]:
    # identical to notebook 02, section 5
    refs, seen = [], {}
    for c in items:
        lender = (c["persons_entitled"] or ["?"])[0]
        lender = lender if len(lender) <= 28 else lender[:27].rstrip() + "…"
        base = c["charge_code"] or f"created {c['created_on']} · {lender}"
        seen[base] = seen.get(base, 0) + 1
        refs.append(base if seen[base] == 1 else f"{base} #{seen[base]}")
    return refs


def charge_records(charges: dict) -> dict[str, dict]:
    # ref -> the fields a reader needs to judge a charge clause
    keep = ("status", "created_on", "persons_entitled", "contains_fixed_charge",
            "contains_floating_charge", "contains_negative_pledge", "particulars")
    items = charges["items"]
    return {r: {"charge": r, **{k: c[k] for k in keep}} for r, c in zip(charge_refs(items), items)}


def decided(applies: bool, evidence=None) -> dict:
    return {"verdict": "applies" if applies else "not_applicable",
            "evidence": list(evidence or []) if applies else [], "by": "code"}

def unknown(why: str) -> dict:
    return {"verdict": "not_determinable", "why": why, "by": "code"}

def read(targets: dict[str, dict]) -> dict:
    return {"verdict": "needs_reading", "targets": targets}


def _listed(section: str, key: str):
    # the clause applies if the signal list is non-empty; the list is the evidence
    return lambda s, recs: decided(bool(s[section][key]), s[section][key])


def _flag_rule(key: str):
    # SEC-07 / SEC-08: the flag settles it where recorded; unrecorded flags on live charges need reading
    def rule(s, recs):
        if s["charges"][key]:
            return decided(True, s["charges"][key])
        unrecorded = s["charges"]["flags_not_recorded_live"]
        return read({r: recs[r] for r in unrecorded}) if unrecorded else decided(False)
    return rule


def sec_06(s, recs):
    groups = s["charges"]["same_lender_within_30d"]
    return decided(bool(groups), [f"{', '.join(g['charges'])} ({g['lender']})" for g in groups])

def con_01(s, recs):
    f, w = s["filings"], s["filings"]["worst_days_late_3y"]
    if w and w["days_late"] > 30:
        return decided(True, [f"{w['ref']} — {w['days_late']} days late"])
    if f["window_truncated"]:
        return unknown("filing window truncated; late accounts earlier in the 3 years may be missing")
    return decided(False)

def con_02(s, recs):
    la = s["filings"]["latest_accounts"]
    late = bool(la and la["days_late"] is not None and la["days_late"] > 180)
    return decided(late, la and [f"{la['ref']} — {la['days_late']} days late"])

def con_03(s, recs):
    f = s["filings"]
    if len(f["late_3y"]) >= 2:
        return decided(True, f["late_3y"])
    if f["window_truncated"]:
        return unknown("filing window truncated; only part of the 3 years was retrieved")
    return decided(False)

def con_04(s, recs):
    f = s["filings"]
    return decided(f["latest_micro_or_exempt"],
                   f["latest_accounts"] and [f"{f['latest_accounts']['ref']} — {f['latest_accounts']['description']}"])

def con_05(s, recs):
    f = s["filings"]
    if f["accounts_on_record"] > 0:
        return decided(False)
    if f["window_truncated"]:
        return unknown("no accounts in the retrieved window, but the window was truncated")
    if f["months_since_incorporation"] is None:
        return unknown("incorporation date missing")
    m = f["months_since_incorporation"]
    return decided(m > 21, [f"no accounts filing; incorporated {m} months before as_of"])

def con_06(s, recs):
    f = s["filings"]
    m = f["months_since_made_up"]
    return decided(m is not None and m > 18, [f"accounts made up to {f['last_made_up_to']} — {m} months before as_of"])

def con_07(s, recs):
    ev = s["filings"]["insolvency_filings"] + (
        ["profile: has_insolvency_history"] if s["company"]["has_insolvency_history"] else [])
    return decided(bool(ev), ev)


RULES = {
    "SEC-01": _listed("charges", "outstanding_third_party"),
    "SEC-02": lambda s, recs: decided(s["charges"]["clean_position"],
                                      s["charges"]["satisfied"] or ["no charges registered"]),
    "SEC-03": _listed("charges", "satisfied"),
    "SEC-04": _listed("charges", "part_satisfied"),
    "SEC-05": _listed("charges", "third_party_last_180d"),
    "SEC-06": sec_06,
    "SEC-07": _flag_rule("negative_pledge"),
    "SEC-08": _flag_rule("floating_live"),
    "CON-01": con_01, "CON-02": con_02, "CON-03": con_03, "CON-04": con_04,
    "CON-05": con_05, "CON-06": con_06, "CON-07": con_07,
    "EVD-01": lambda s, recs: decided(bool(s["missing"]), s["missing"]),
}

ids = {c["clause_id"] for c in CLAUSES}
print(f"{len(RULES)} rules in code · {len(ROUTED)} routed · "
      f"{len(ids - set(RULES) - set(ROUTED))} left for the model: {sorted(ids - set(RULES) - set(ROUTED))}")
assert not (set(RULES) | set(ROUTED)) - ids, "a rule or route names a clause that isn't in the corpus"

16 rules in code · 7 routed · 0 left for the model: []


## 6 · The model — only for what needs reading

Same short leash as notebook 02, tightened for a decision:

1. **Three answers, one of them "I can't tell".** `not_determinable` is a first-class answer. The prompt says
   to use it whenever the record is silent or incomplete, e.g. particulars that end *"see image for full
   details"*. A bank's policy check must never turn "not stated" into "not applicable".
2. **Every `applies` / `not_applicable` must quote the record**, and code checks each quote is really there.
   An answer with a quote that can't be found is downgraded to `not_determinable`, so an unverifiable answer
   becomes an evidence gap, not a decision.

   `quotes` is a **list** because of a failure found while building this notebook. With a single `quote`, the
   model answered SEC-09 (section 12) correctly but joined two values into one quote:
   `'"status": "outstanding" ... "Tesco Ireland Pension Trustees Limited as Trustee of…"'`. That string isn't
   in the record, so the check rejected a correct answer. Evidence that spans two fields is normal. Asking for
   one quote per value, each checked on its own, fixes the format without loosening the check.
3. **The model never sees or sets an outcome.** It answers "does this clause apply?". The outcome (REFER,
   DECLINE…) is read from the clause's own `Outcome:` line, and precedence is code.
4. **One call per company**, all questions batched, and no call when nothing needs reading.

Effort is left at the default (high), unlike 02's `low`: this answer feeds a lending decision, not a label.

In [13]:
MODEL = "claude-sonnet-5"
llm = anthropic.AsyncAnthropic()


class Judgement(BaseModel):
    id: str
    verdict: Literal["applies", "not_applicable", "not_determinable"]
    quotes: list[str] = Field(description="Words copied exactly from single values in the record that show the "
                                          "verdict, one entry per value. Empty if not_determinable.")
    reason: str = Field(description="One sentence, citing only the record.")


class Judgements(BaseModel):
    judgements: list[Judgement]


SYSTEM = (
    "You check whether a lending-policy clause applies, using only the record given with each question. "
    "Answer applies only if the record shows the clause's condition is met, and not_applicable only if the "
    "record shows it is not met. If the record is silent, incomplete, or points to a document you do not have "
    "(for example 'see image for full details'), answer not_determinable and say what is missing. "
    "Do not assume, and do not use knowledge about lenders, companies or what such charges usually contain. "
    "For applies and not_applicable, give quotes: each one copied exactly from a single value in the record, "
    "with no field names, JSON punctuation or '...'. Use several quotes when the evidence is in several values."
)


def _norm(s: str) -> str:
    return " ".join(s.lower().split())

def _flatten(x) -> str:
    if isinstance(x, dict):
        return " ".join(_flatten(v) for v in x.values())
    if isinstance(x, list):
        return " ".join(_flatten(v) for v in x)
    return "" if x is None else str(x)


async def judge(questions: list[dict]) -> dict[str, dict]:
    if not questions:
        print("policy: nothing needs reading — 0 model calls")
        return {}
    payload = [{"id": q["id"], "clause": q["clause_text"], "record": q["record"]} for q in questions]
    resp = await llm.beta.messages.parse(
        model=MODEL,
        max_tokens=16000,
        system=SYSTEM,
        messages=[{"role": "user", "content": json.dumps(payload, indent=1)}],
        output_format=Judgements,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )
    print(f"policy: 1 call · {len(questions)} question(s) · {resp.model} · "
          f"{resp.usage.input_tokens} in / {resp.usage.output_tokens} out tokens")

    ok = resp.stop_reason != "refusal" and resp.parsed_output is not None
    got = {j.id: j for j in resp.parsed_output.judgements} if ok else {}
    out = {}
    for q in questions:
        j = got.get(q["id"])
        if j is None:
            out[q["id"]] = {"verdict": "not_determinable", "quotes": [], "reason": "no answer from the model"}
            continue
        flat = _norm(_flatten(q["record"]))
        missing = [x for x in j.quotes if _norm(x) not in flat]
        if j.verdict != "not_determinable" and (not j.quotes or missing):
            out[q["id"]] = {"verdict": "not_determinable", "quotes": j.quotes,
                            "reason": f"quote not found in the record: {missing or 'none given'} "
                                      f"(model said {j.verdict}: {j.reason})"}
        else:
            out[q["id"]] = {"verdict": j.verdict, "quotes": j.quotes, "reason": j.reason}
    return out

## 7 · Precedence (EVD-07) and `qualifies`

Clause outcomes are free text in the policy files ("REFER", "INSUFFICIENT EVIDENCE — return for research",
"PROCEED WITH QUALIFICATION"). `_level` maps each to one of the four EVD-07 levels, and the decision is the
most restrictive level among the clauses that apply. Every clause at that level is recorded as having
decided it, since EVD-06 requires the brief to name them.

In [14]:
RANK = {"PROCEED": 0, "REFER": 1, "INSUFFICIENT EVIDENCE": 2, "DECLINE": 3}      # EVD-07


def _level(outcome_text: str) -> str | None:
    t = outcome_text.upper()
    return next((k for k in sorted(RANK, key=len, reverse=True) if t.startswith(k)), None)


def decide(applicable: list[dict], gaps: list[str]) -> tuple[dict, bool | None]:
    levels = [_level(a["outcome"]) for a in applicable if _level(a["outcome"])]
    decision = max(levels, key=RANK.get, default="PROCEED")
    outcome = {"decision": decision,
               "clauses": [a["clause_id"] for a in applicable if _level(a["outcome"]) == decision]}
    qualifies = False if decision == "DECLINE" else (None if gaps else True)
    return outcome, qualifies

## 8 · The policy node

`evaluate` runs every clause in order: routed → recorded, rule → run it, no rule → send the clause to the
model with the company's full signals. The questions are then batched into one call and the answers folded
back per clause. A clause applies if it applies to any target (any charge); otherwise it's undetermined if any
target was; otherwise not applicable.

It takes the clause list as an argument, not the global, so section 12 can pass an extra clause in without
touching the policy files.

In [15]:
async def evaluate(signals: dict, charges: dict, clauses: list[dict]) -> dict:
    recs = charge_records(charges)
    assert set(signals["charges"]["live"]) <= set(recs), "charge refs differ from notebook 02's charge_refs"

    results, questions = {}, []
    for c in clauses:
        cid = c["clause_id"]
        if cid in ROUTED:
            results[cid] = {"verdict": f"routed → {ROUTED[cid]}"}
            continue
        if cid in RULES:
            r = RULES[cid](signals, recs)
        else:
            print(f"(!) {cid} has no rule in code — the model judges it from the clause text")
            r = read({"company": {"signals": signals, "charges": list(recs.values())}})
        if r["verdict"] == "needs_reading":
            for target, record in r["targets"].items():
                questions.append({"id": f"q{len(questions) + 1}", "clause_id": cid, "target": target,
                                  "clause_text": c["text"], "record": record})
        results[cid] = r

    answers = await judge(questions)
    for cid in {q["clause_id"] for q in questions}:
        qa = [(q, answers[q["id"]]) for q in questions if q["clause_id"] == cid]
        hits = [f"{q['target']}: " + " + ".join(f"«{x}»" for x in a["quotes"])
                for q, a in qa if a["verdict"] == "applies"]
        unk  = [f"{q['target']}: {a['reason']}" for q, a in qa if a["verdict"] == "not_determinable"]
        results[cid] = ({"verdict": "applies", "evidence": hits, "by": "model"} if hits else
                        {"verdict": "not_determinable", "why": " | ".join(unk), "by": "model"} if unk else
                        {"verdict": "not_applicable", "evidence": [], "by": "model"})

    applicable, gaps, trace = [], [], {}
    for c in clauses:
        cid, r = c["clause_id"], results[c["clause_id"]]
        trace[cid] = r["verdict"] + (f" · {r['by']}" if "by" in r else "")
        if r["verdict"] == "applies":
            applicable.append({"clause_id": cid, "title": c["title"], "outcome": c["outcome"],
                               "evidence": r["evidence"], "decided_by": r["by"]})
        elif r["verdict"] == "not_determinable":
            gaps.append(f"{cid} ({_level(c['outcome']) or c['outcome']}) not determinable — {r['why']}")
    gaps += [f"EVD-01 missing: {m}" for m in signals["missing"]]

    outcome, qualifies = decide(applicable, gaps)
    return {"applicable": applicable, "outcome": outcome, "qualifies": qualifies,
            "evidence_gap": gaps, "policy_trace": trace}


async def policy(state: CopilotState) -> dict:
    return await evaluate(state["signals"], state["charges"], CLAUSES)

## 9 · Build the graph

`START → policy → END`, run on the saved state. Research and signal already ran in notebook 02; their
output *is* the input here. The full chain `research → signal → policy` is wired in notebook 04, once the
nodes live in a module they can all import (see *What comes next*).

In [16]:
g = StateGraph(CopilotState)
g.add_node("policy", policy)
g.add_edge(START, "policy")
g.add_edge("policy", END)
graph = g.compile()


def show(st: dict) -> None:
    o = st["outcome"]
    print(f"decision : {o['decision']}  (set by {', '.join(o['clauses']) or '—'})")
    print(f"qualifies: {st['qualifies']}\n")
    print("applicable:")
    for a in st["applicable"]:
        print(f"  {a['clause_id']}  {_level(a['outcome']) or a['outcome']:22s} {a['decided_by']:5s}  {a['evidence']}")
    print("\nevidence_gap:" + ("" if st["evidence_gap"] else " none"))
    for gline in st["evidence_gap"]:
        print("  " + gline)
    counts = {}
    for v in st["policy_trace"].values():
        counts[v.split(" · ")[0].split(" →")[0]] = counts.get(v.split(" · ")[0].split(" →")[0], 0) + 1
    print(f"\ntrace: {len(st['policy_trace'])} clauses checked — {counts}")

## 10 · Run it — 36EL LTD (`10812571`)

In [17]:
r36 = await graph.ainvoke(FIXTURES["10812571"])
show(r36)

policy: nothing needs reading — 0 model calls
decision : DECLINE  (set by CON-02)
qualifies: False

applicable:
  CON-01  REFER                  code   ['AA filed 2025-12-17 — 264 days late']
  CON-02  DECLINE                code   ['AA filed 2025-12-17 — 264 days late']
  CON-03  REFER                  code   ['AA filed 2025-12-17', 'AA filed 2024-06-27']
  CON-04  REFER                  code   ['AA filed 2025-12-17 — accounts with accounts type micro entity (made up date: 2024-06-30)']
  CON-06  REFER                  code   ['accounts made up to 2024-06-30 — 26.8 months before as_of']
  SEC-01  REFER                  code   ['108125710002', '108125710001']
  SEC-06  REFER                  code   ['108125710001, 108125710002 (interbay funding limited)']
  SEC-07  REFER                  code   ['108125710002', '108125710001']
  SEC-08  REFER                  code   ['108125710002', '108125710001']

evidence_gap: none

trace: 23 clauses checked — {'applies': 9, 'not_applicable': 7, 'ro

What to notice:

- **The decision is DECLINE, set by CON-02 alone.** Eight other clauses apply (all REFER), and they're all kept in
  `applicable` because EVD-07 says every triggered clause must still be cited. Precedence picks the decision;
  it doesn't discard the rest.
- **Zero model calls.** 36EL's charges are post-2013, so every flag is recorded, and its filing window wasn't
  truncated. The record settled every clause.
- **`qualifies` is `False`, not `None`,** even if a gap existed: nothing missing could make a DECLINE less restrictive.

## 11 · Run it — TESCO PLC (`00445790`): the model's first real job

Tesco's two live charges are from 2009, so their pledge and floating flags were never recorded (SEC-07, SEC-08 → read).
Its filing history was truncated, so the 3-year lateness counts can't be complete (CON-01, CON-03 → unknown).

In [18]:
rt = await graph.ainvoke(FIXTURES["00445790"])
show(rt)

policy: 1 call · 4 question(s) · claude-sonnet-5 · 2449 in / 322 out tokens
decision : REFER  (set by SEC-01)
qualifies: None

applicable:
  SEC-01  REFER                  code   ['created 2009-11-04 · Tesco Trustee Company of Ir…', 'created 2009-11-04 · Tesco Ireland Pension Trust…']
  SEC-03  PROCEED                code   ['created 2009-03-27 · Tesco Ireland Pension Trust…', 'created 2009-03-27 · Tesco Trustee Company of Ir…', 'created 2005-05-13 · Rbs Aerospace Limited', 'created 2001-07-31 · Deutsche International Fina…', 'created 2000-12-08 · Deutsche Bank Ag', 'created 1994-04-05 · Cobroad Investments', 'created 1991-12-17 · Cobroad Investments']

evidence_gap:
  CON-01 (REFER) not determinable — filing window truncated; late accounts earlier in the 3 years may be missing
  CON-03 (REFER) not determinable — filing window truncated; only part of the 3 years was retrieved
  SEC-07 (REFER) not determinable — created 2009-11-04 · Tesco Trustee Company of Ir…: contains_negative_pledge

## 12 · A clause with no rule — why the model path exists at all

The policy files are written by credit risk, not by whoever maintains this notebook. When a new clause appears,
the node must not ignore it until someone writes a rule. So a clause with no rule goes to the model, with the
company's full signals and charge records.

To test that path without editing the real policy files, here is a hypothetical clause, written in the same
format, passed in alongside the real 23:

In [19]:
SEC_09 = {
    "clause_id": "SEC-09",
    "doc": "SEC",
    "title": "SEC-09 — Charge in favour of a pension scheme trustee",
    "outcome": "REFER",
    "text": (
        "### SEC-09 — Charge in favour of a pension scheme trustee\n**Outcome: REFER**\n\n"
        "A live charge in favour of the trustee of a pension scheme indicates the company has granted security "
        "to its pension scheme. Refer for assessment of the scheme's claim on the company's assets.\n\n"
        "*Applies when:* any charge that is not fully satisfied names a pension scheme trustee in `persons_entitled`."
    ),
}

demo = await evaluate(FIXTURES["00445790"]["signals"], FIXTURES["00445790"]["charges"], CLAUSES + [SEC_09])
print("\ntrace SEC-09:", demo["policy_trace"]["SEC-09"])
for a in demo["applicable"]:
    if a["clause_id"] == "SEC-09":
        for e in a["evidence"]:
            print("  ", e)
for gline in demo["evidence_gap"]:
    if gline.startswith("SEC-09"):
        print("  ", gline)
print(f"\ndecision with SEC-09: {demo['outcome']}   (without: {rt['outcome']})")

(!) SEC-09 has no rule in code — the model judges it from the clause text
policy: 1 call · 5 question(s) · claude-sonnet-5 · 5818 in / 593 out tokens

trace SEC-09: applies · model
   company: «outstanding» + «Tesco Trustee Company of Ireland Limited as Trustee of the Tesco Ireland Limited Senior Executive Pension Scheme» + «outstanding» + «Tesco Ireland Pension Trustees Limited as Trustee of the Tesco Ireland Limited Pension Plan»

decision with SEC-09: {'decision': 'REFER', 'clauses': ['SEC-01', 'SEC-09']}   (without: {'decision': 'REFER', 'clauses': ['SEC-01']})


No rule was written for SEC-09, yet it was applied. The model gave three quotes: the status `outstanding`
and the two trustee names from `persons_entitled`. Code found each one in the record. SEC-09 is REFER, so the
decision stays REFER, now set by SEC-01 *and* SEC-09. Both would be cited in the brief.

**One limit of the quote check, visible here:** it proves each quote is *in* the record, not that the quotes
belong *together*. `«outstanding»` could in principle come from a different charge than the trustee names.
Here it doesn't (both trustee charges are the live ones, see SEC-01 above), but the check alone doesn't prove
that. Asking for quotes per charge, not per company, would close that gap.

That is the one thing the model does in this node that code can't: apply a clause nobody has turned into code
yet. Once a clause matters often enough, writing its rule makes it free and deterministic, and the model path
goes quiet for it.

## 13 · Why this node doesn't use similarity search

`policy_store.py` was built to feed this node. `06_rag.ipynb` found that one query per fact cluster
(`retrieve_facets`) beats one blended query. But selection by similarity has to be *complete* to be safe: a clause
that isn't retrieved is a clause that is never checked. Here are 06's own faceted queries for 36EL, scored
against the clauses the rules above found to apply:

In [20]:
from policy_store import retrieve_facets

FACETS_06 = {   # verbatim from 06_rag.ipynb, section 4 — written for 36EL
    "charges":  "two charges outstanding in favour of a third-party lender, neither satisfied",
    "accounts": "micro-entity accounts filed 8 months after the statutory deadline",
    "evidence": "is there enough evidence to make a recommendation",
}
need = {a["clause_id"] for a in r36["applicable"]}
for k in (3, 5, 8):
    got = {h["clause_id"] for h in retrieve_facets(FACETS_06, k=k)}
    print(f"k={k}: retrieved {len(got):2d} of {len(CLAUSES)} clauses · found {len(need & got)} of {len(need)} "
          f"that apply · missed {sorted(need - got)}")

/Users/natchalin_/Projects/final_project/Lloyds/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18213.81it/s]


k=3: retrieved  9 of 23 clauses · found 4 of 9 that apply · missed ['CON-02', 'CON-03', 'CON-06', 'SEC-07', 'SEC-08']
k=5: retrieved 15 of 23 clauses · found 6 of 9 that apply · missed ['CON-03', 'SEC-07', 'SEC-08']
k=8: retrieved 20 of 23 clauses · found 8 of 9 that apply · missed ['SEC-08']


- **At k=3, CON-02 is missed**, the clause that makes 36EL a DECLINE. A retrieval-first policy node would have
  scored this company REFER, the wrong outcome.
- **At k=8, retrieval returns almost the whole corpus** and still misses a clause. It's no longer selecting
  anything, just adding a way to fail.

So selection here is by clause ID over the whole corpus: every clause is checked, every time, in code or by
the model. Retrieval keeps its place where the question really is "which of thousands of passages is relevant?".
At 23 clauses with an ID each, it isn't. This is the "honest alternative at this corpus size" that 06
anticipated, now with numbers.

## What comes next

| notebook | node | reads | adds to state |
|---|---|---|---|
| 04 | `supervisor` | `evidence_gap`, a retry counter | routes back to research, or on to brief; after two retries, a "thin evidence" brief (EVD-05) |
| 04 | `brief` | `profile`, `signals`, `applicable`, `outcome` | the RM brief, citing clause IDs (EVD-06) and records (EVD-02); collateral only where `verified` (EVD-04) |

Three things to settle before or in 04:

- **Move the nodes into a module** (like `policy_store.py` came out of 06), so 04 can import `research`,
  `signal` and `policy` and wire the full graph. That also removes the copied `charge_refs`.
- **A retry won't fix every gap.** Tesco's gaps come from `max_items=60` in `get_filing_history` and from
  pre-2013 charges whose full instrument is only an image. Running research again returns the same record,
  so the supervisor should tell "retry might help" apart from "retry can't help", or it will spend both of
  EVD-05's retries for nothing.
- **CON-08** still needs `paper_filed` in `mcp_ch/tools.py`, if the brief is to mention paper filing at all.